モデルを学習させ、PTH形式で保存する

In [1]:
from accelerate import Accelerator
from torch.optim import AdamW
from tqdm.notebook import tqdm
from bitsandbytes import optim as bnb_optim
import torch
import torch.nn as nn

def train(model, dataloader, vocab_size, epochs=3, lr=3e-4, weight_decay=0.0, early_stop_loss=0.1):
    accelerator = Accelerator()
    device = accelerator.device

    model.to(device)
    optimizer = bnb_optim.AdamW8bit(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)
    model.train()
    for epoch in range(epochs):
        pbar = tqdm(dataloader, desc=f"Epoch: {epoch+1}", disable=not accelerator.is_local_main_process)
        for batch in pbar:
            input_ids = batch[:, :-1]
            labels = batch[:, 1:]

            outputs = model(input_ids)

            loss = criterion(outputs.reshape(-1, vocab_size), labels.reshape(-1))


            optimizer.zero_grad()
            accelerator.backward(loss)
            optimizer.step()

            pbar.set_postfix(loss=loss.item())

            if loss.item() < early_stop_loss:
                print(f"Training stopped early at epoch {epoch+1}, batch {pbar.n} due to train_loss < {early_stop_loss}")
                return  # 訓練を中止
        
        torch.save(model.state_dict(), f"../data/model_{path.max_seq_len}.pth")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import path as path
from dataset import JPNDataset
import freqModel

dataset = torch.load(f"../data/dataset_{path.max_seq_len}.pt", weights_only=False)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

model = freqModel.Decoder(vocab_size=22287, d_model=64, n_heads=4, n_layers=2, max_seq_len=path.max_seq_len, dropout=0.0)
train(model, dataloader, vocab_size=22287, epochs=3, lr=3e-4, weight_decay=0.01)

Epoch: 1:   0%|          | 0/96741 [00:00<?, ?it/s]

推論

In [ ]:
import torch
import json
import path as path
from charTokenizer import CharTokenizer

model = Decoder(vocab_size=22217, d_model=128, n_heads=4, n_layers=2, max_seq_len=path.max_seq_len, dropout=0.0)
model.load_state_dict(torch.load(f"../data/model_{path.max_seq_len}.pth"))
model.eval()

tokenizer = CharTokenizer()
with open(path.charTokenizer, 'r', encoding='utf-8') as f:
    tokenizer.vocab = json.load(f)
inputs = tokenizer(
    "荒れた内を避ける為か中間付近までは馬場の中央付近を走行。道中ペースを緩め脚をためると、そのまま直線も先頭でゴールした。逃げての上がりは32.9で上がり最速タイ。 武豊騎手は「ポンと出て、無理に引っ張ることもなく、マイペースで行けた。ラストでひと伸びして能力の高さを感じた」とコメント。ききょうステークス以来実に1年ぶりの勝利を飾り、素質の高さを見せた。続く逆瀬川ステークスでは前走の走りを評価されてか、古馬と同じ55kgの斤量を課された。チャンピオンズカップの裏開催であった為、武豊騎手から吉田隼人騎手に乗り替わり、朝日杯FS以来のコンビ結成となった。スムーズにゲートを出ると、そのまま内3番手を追走。最後直線は力強く抜け出して、2連勝でのOP入りを決めた。吉田隼人騎手は「約1年ぶりに乗せていただきましたが成長しています。出たなりでいい位置をキープできました。抜け出してから、左にもたれる癖はあるが、上がり勝負にも対応してくれました。競馬に幅が広がったし、これからが楽しみです」と振り返った。2023年（4歳）.明け4歳の始動戦に選ばれたのは東京芝2000mのリステッド戦である[白富士ステークスとされた。当日は前走ローズステークス2着と好走したサリエラに次ぐ2番人気に評価された。レースはドーブネが最内枠から好スタートでハナを奪い、武豊のエスコートで1000m59.9秒という絶妙な時計で逃げを打つ。直線に入り粘りの逃げで後続を離すかに思えたが、残り200mあたりから失速。最後は後方から末脚を伸ばしてきたサリエラに交わされた。それでも内を通って伸びてきていたヤマニンサルバムには抜かせず2着を確保した。レース後武豊は「楽なペースだったけどね…。1ハロンぐらい少し距離が長いのかな」とコメントを残し、2000ｍはドーブネにとって距離が長い可能性が示唆された。"
    , max_seq_len=path.max_seq_len, return_tensors="pt")
with torch.no_grad():
    outputs = model(inputs["input_ids"])
    predictions = torch.argmax(outputs, dim=-1)
output_text = tokenizer.decode(predictions[0])
print(output_text)

In [ ]:
import torch.nn.functional as F

with torch.no_grad():
    x = torch.randn(64, 64, 64)
    x_fft = torch.fft.fft(x, dim=1)
    x_ifft = torch.fft.ifft(x_fft, dim=1).real
    mse = F.mse_loss(x, x_ifft)
    print("FFT-IFFT再構成誤差:", mse.item())


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import model
import freqModel

def check_model_learns(model, vocab_size, seq_len=16, batch_size=4, epochs=100, lr=3e-4):
    model.train()
    device = next(model.parameters()).device

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        # ランダムな自然数列を生成 (0 ~ vocab_size-1)
        x = torch.randint(low=0, high=vocab_size, size=(batch_size, seq_len), device=device)
        # 目標は次トークン予測なのでシフトしたもの
        y = torch.roll(x, shifts=-1, dims=1)

        optimizer.zero_grad()
        logits = model(x)  # [B, T, vocab_size]

        # CrossEntropyLoss用に形状を変換
        logits_flat = logits.view(-1, vocab_size)  # [B*T, vocab_size]
        y_flat = y.view(-1)  # [B*T]

        loss = criterion(logits_flat, y_flat)
        loss.backward()
        optimizer.step()

        print(f"Epoch {epoch+1}: loss={loss.item():.4f}")

# モデル定義（例）
vocab_size = 50
d_model = 64
n_heads = 4
n_layers = 2
max_seq_len = 256

model = freqModel.Decoder(vocab_size, d_model, n_heads, n_layers, max_seq_len).to('cuda' if torch.cuda.is_available() else 'cpu')
check_model_learns(model, vocab_size=vocab_size, seq_len=max_seq_len)
